# experiment.ipynb

## Imports

In [ ]:
! pip install matplotlib

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) 
sys.path.append(project_root)

print("Sys path:", sys.path)

In [ ]:
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from PIL import Image
import torch
import numpy as np
import random

from data.dataset import AIGCDataset
from util.image_util import train_transform, val_transform

from preprocess.smash import smash
from preprocess.reconstruct import reconstruct
from model.filters import SRMFilters
from model.fingerprint import FingerprintExtractor 

## Data

In [ ]:
train_root = r"C:\Users\seths\VSCode\AI-generated_Image_detection_algorithm\data\datasets\anhphmminh\cnnspot\versions\1\cnn_spot\train"

train_ds = AIGCDataset(root_dir = train_root, transform = train_transform, split = 'train')

train_loader = DataLoader(train_ds, batch_size = 16, shuffle = False, num_workers = 16)

In [ ]:
img, lbl = train_ds[0]
print("Single image shape:", img.shape)
print("Label:", lbl)

In [ ]:
images, labels = next(iter(train_loader))
fig, axs = plt.subplots(4, 4, figsize=(12, 12))

for i, ax in enumerate(axs.flat):
    img_denorm = (images[i] * 0.5) + 0.5
    img_denorm = img_denorm.clamp(0, 1)
    img_denorm = img_denorm.permute(1, 2, 0).cpu().numpy()
    ax.imshow(img_denorm)
    ax.set_title(f"Label: {'Fake' if labels[i] else 'Real'}")
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Toy image creator: Simple gradient for testing (256x256 RGB)
def create_toy_image(size=256, mode='gradient'):
    if mode == 'gradient':
        x = np.linspace(0, 1, size)
        y = np.linspace(0, 1, size)
        r, g = np.meshgrid(x, y)
        b = np.zeros_like(r)
        img_array = np.stack([r, g, b], axis=-1) * 255
    elif mode == 'noise':
        img_array = np.random.randint(0, 256, (size, size, 3))
    elif mode == 'texture_mix':
        img_array = np.zeros((size, size, 3))
        img_array[:size//2, :] = 128  # Smooth half
        img_array[size//2:, :] = np.random.randint(0, 256, (size//2, size, 3))  # Noisy half
    return Image.fromarray(img_array.astype(np.uint8))

# Device for models
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Smash

Purpose: Verify patch extraction, diversity scoring (Eq. 1 from PatchCraft), sorting, and selection of rich/poor patches.

Input: Toy image (smooth + noisy for clear diversity).

Expected: Rich patches have high variance; poor have low. Shapes: Two tensors of [K, 3, 32, 32].

In [ ]:
# Test Smash
toy_img = create_toy_image(mode='texture_mix')  # Half smooth, half noisy for testing diversity
rich_patches, poor_patches = smash(toy_img)

# Visual check: Plot a few patches
fig, axs = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    axs[0, i].imshow(rich_patches[i].permute(1, 2, 0).numpy())
    axs[0, i].set_title("Rich Patch")
    axs[1, i].imshow(poor_patches[i].permute(1, 2, 0).numpy())
    axs[1, i].set_title("Poor Patch")
    axs[0, i].axis('off')
    axs[1, i].axis('off')
plt.show()

# Asserts (basic shape checks)
assert rich_patches.shape == poor_patches.shape, "Rich and poor should have same shape"
assert rich_patches.shape[1] == 3, "Should be RGB (3 channels)"
print("Smash test passed if rich patches look noisier than poor.")

## Test Reconstruct Function

Purpose: Verify collage assembly from patches, retaining boundaries (PatchCraft Sec. 3.2).

Input: Set of patches from Smash.

Expected: Square collage tensor (e.g., [3, 256, 256]); boundaries visible.


In [ ]:
# Test Reconstruct (use patches from Smash test)
rich_collage = reconstruct(rich_patches)
poor_collage = reconstruct(poor_patches)

# Visual check
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(rich_collage.permute(1, 2, 0).numpy())
axs[0].set_title("Rich Collage")
axs[1].imshow(poor_collage.permute(1, 2, 0).numpy())
axs[1].set_title("Poor Collage")
plt.show()

# Asserts
assert rich_collage.shape == (3, 256, 256), "Collage should be [3, 256, 256]"
print("Reconstruct test passed if collages show patch boundaries.")

## Test Filters Function

Purpose: Verify SRM high-pass filters compute residuals (Rich Models Eq. 1-2, Fig. 2).

Input: Collage tensor from Reconstruct.

Expected: Stack of 30 residual maps [1, 30, H, W]; values small/centered around 0 (noise-like).


In [ ]:
# Test Filters (use collage from Reconstruct test)
filters = SRMFilters().to(device)  # Init the module
collage_tensor = rich_collage.unsqueeze(0).to(device)  # [1, 3, 256, 256]
residuals = filters(collage_tensor)

# Visual check: Plot a few residuals
fig, axs = plt.subplots(3, 5, figsize=(15, 9))
for i, ax in enumerate(axs.flat):
    if i < residuals.shape[1]:
        res = residuals[0, i].cpu().numpy()
        ax.imshow(res, cmap='gray')
        ax.set_title(f"Residual {i}")
        ax.axis('off')
plt.show()

# Asserts
assert residuals.shape[1] == 30, "Should output 30 residuals"
assert torch.mean(torch.abs(residuals)) < 50, "Residuals should be small values (noise)"
print("Filters test passed if residuals look like noise patterns.")

## Test Fingerprint Extraction Function

Purpose: Verify processing residuals and computing contrast (PatchCraft Sec. 3.2: conv + BN + HardTanh + subtract).

Input: Rich and poor residuals from Filters.

Expected: Feature vector [1, feat_dim] (e.g., 30); values in [-1,1] from HardTanh.

In [ ]:
# Test Fingerprint (use residuals from Filters test)
extractor = FingerprintExtractor().to(device)
rich_res = filters(rich_collage.unsqueeze(0).to(device))
poor_res = filters(poor_collage.unsqueeze(0).to(device))
fingerprint = extractor(rich_res, poor_res)

# Visual check: Plot the feature vector
plt.figure(figsize=(10, 3))
plt.bar(range(len(fingerprint[0])), fingerprint[0].cpu().numpy())
plt.title("Fingerprint Feature Vector")
plt.xlabel("Feature Dim")
plt.ylabel("Value")
plt.show()

# Asserts
assert fingerprint.shape[1] == 30, "Feature dim should match input channels"
assert torch.all((fingerprint >= -1) & (fingerprint <= 1)), "HardTanh should clip to [-1,1]"
print("Fingerprint test passed if vector shows contrast differences.")